# Notebook 03 — Evaluate All Experiments
**Member: C** | Set `model.checkpoint_path` in `configs/eval.yaml` before each run.

| Experiment | `checkpoint_path` | Benchmark |
|---|---|---|
| Exp 0 zero-shot    | `null`                             | GSM8K |
| Exp 1 few-shot CoT | `null`                             | GSM8K |
| Exp 2 adaptation   | `results/checkpoints/adaptation`   | GSM8K |
| Exp 3 distillation | `results/checkpoints/distillation` | GSM8K |
| Exp 4 filtered     | `results/checkpoints/filtered`     | GSM8K |
| Exp 5 cross-task   | `results/checkpoints/filtered`     | ARC |

In [2]:

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/SC4001")
RESULTS = BASE / "results"

print("SC4001 folder found:", BASE.is_dir())
print("Results folder found:", RESULTS.is_dir())


Mounted at /content/drive
SC4001 folder found: True
Results folder found: True


In [3]:

from pathlib import Path

existing_files = [
    "exp5_arc_exp3_updated_600tokens_arc.csv",
    "exp5_arc_exp4_updated_600tokens_arc.csv"
]

print("=== EXISTING ARC RESULTS ===")

for filename in existing_files:
    path = RESULTS / filename
    print(filename, ":", "FOUND" if path.is_file() else "NOT FOUND")

print("\n=== BASELINE RESULTS ===")

for path in sorted(RESULTS.glob("*baseline*arc*")):
    print("Existing:", path.name)

print("\nSafety check finished. No files changed.")


=== EXISTING ARC RESULTS ===
exp5_arc_exp3_updated_600tokens_arc.csv : FOUND
exp5_arc_exp4_updated_600tokens_arc.csv : FOUND

=== BASELINE RESULTS ===

Safety check finished. No files changed.


In [4]:

import subprocess
from pathlib import Path

REPO = Path("/content/SC4001_ARC_BASELINE_REVIEW")

if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/TIFFMUN/SC4001-Group133.git",
        str(REPO)
    ], check=True)

assert (REPO / ".git").exists(), "GitHub clone not available"

print("Repository:", REPO)
print("ARC evaluator exists:",
      (REPO / "src/evaluation/eval_arc.py").is_file())
print("Evaluation config exists:",
      (REPO / "configs/eval.yaml").is_file())

print("Commit:",
      subprocess.check_output(
          ["git", "-C", str(REPO), "rev-parse", "HEAD"],
          text=True
      ).strip())


Repository: /content/SC4001_ARC_BASELINE_REVIEW
ARC evaluator exists: True
Evaluation config exists: True
Commit: d79ef83ec844bedf133b460b1ab59b692f0fe62d


In [5]:

import yaml
from pathlib import Path

config_file = REPO / "configs/eval.yaml"
script_file = REPO / "src/evaluation/eval_arc.py"

with config_file.open() as f:
    cfg = yaml.safe_load(f)

print("=== MODEL SETTINGS ===")
print(cfg.get("model"))

print("\n=== ARC SETTINGS ===")
print(cfg.get("arc"))

print("\n=== EVALUATION IMPLEMENTATION ===")

lines = script_file.read_text().splitlines()

keywords = [
    "def evaluate_arc",
    "checkpoint_path",
    "from_pretrained",
    "PeftModel",
    "load_model",
    "to_csv",
    "open(",
    "results_dir",
    "output_path"
]

indices = set()

for i, line in enumerate(lines):
    if any(k in line for k in keywords):
        indices.update(range(max(0, i-2), min(len(lines), i+4)))

for i in sorted(indices):
    print(f"{i+1}: {lines[i]}")


=== MODEL SETTINGS ===
{'base_model': 'Qwen/Qwen2.5-0.5B-Instruct', 'checkpoint_path': 'results/checkpoints/distillation'}

=== ARC SETTINGS ===
{'dataset': 'allenai/ai2_arc', 'config': 'ARC-Challenge', 'split': 'test', 'max_samples': None, 'batch_size': 8}

=== EVALUATION IMPLEMENTATION ===
24: import yaml
25: from datasets import load_dataset
26: from peft import PeftModel
27: from transformers import AutoModelForCausalLM, AutoTokenizer
28: from tqdm import tqdm
29: 
34: 
35: 
36: def load_model(cfg: dict, checkpoint_override: str | None = None):
37:     """Load base or LoRA-finetuned model. Same logic as eval_gsm8k.py."""
38:     base_model_name = cfg["model"]["base_model"]
39:     checkpoint_path = checkpoint_override or cfg["model"].get("checkpoint_path")
40: 
41:     # Always safely load tokenizer from base model
42:     tokenizer = AutoTokenizer.from_pretrained(base_model_name)
43:     if tokenizer.pad_token is None:
44:         tokenizer.pad_token = tokenizer.eos_token
45: 
46:

In [6]:

import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/content/SC4001_ARC_BASELINE_REVIEW")

assert (REPO / "src/evaluation/eval_arc.py").is_file(), \
    "Repository not found. Check Cell 3."

# Install dependencies in the Colab runtime
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(REPO / "requirements.txt")
], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-e", str(REPO)
], check=True)

os.chdir(REPO)

import torch
print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Setup ready.")


GPU available: True
GPU: Tesla T4
Setup ready.


In [8]:

import yaml
import tempfile
from pathlib import Path

# New temporary folder, separate from your Google Drive
RUN_DIR = Path(tempfile.mkdtemp(
    prefix="SC4001_ARC_BASELINE_",
    dir="/content"
))

# Read the existing configuration without modifying it
with (REPO / "configs/eval.yaml").open() as f:
    cfg = yaml.safe_load(f)

# Use ORIGINAL pretrained model, without LoRA
cfg["model"]["base_model"] = "Qwen/Qwen2.5-0.5B-Instruct"
cfg["model"]["checkpoint_path"] = None

# Match previous ARC evaluations
cfg["arc"]["dataset"] = "allenai/ai2_arc"
cfg["arc"]["config"] = "ARC-Challenge"
cfg["arc"]["split"] = "test"
cfg["arc"]["max_samples"] = None
cfg["benchmarks"]["arc"] = True

# Save all new outputs to temporary folder
cfg["output"]["results_dir"] = str(RUN_DIR)

SAFE_CONFIG = RUN_DIR / "eval_arc_baseline.yaml"

with SAFE_CONFIG.open("x") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("=== BASELINE CONFIGURATION ===")
print("Model:", cfg["model"])
print("ARC:", cfg["arc"])
print("Output folder:", RUN_DIR)
print("Original configuration unchanged:", True)

assert cfg["model"]["checkpoint_path"] is None
print("\nBASELINE CONFIGURATION READY")


=== BASELINE CONFIGURATION ===
Model: {'base_model': 'Qwen/Qwen2.5-0.5B-Instruct', 'checkpoint_path': None}
ARC: {'dataset': 'allenai/ai2_arc', 'config': 'ARC-Challenge', 'split': 'test', 'max_samples': None, 'batch_size': 8}
Output folder: /content/SC4001_ARC_BASELINE_lp7_9atd
Original configuration unchanged: True

BASELINE CONFIGURATION READY


In [9]:

import sys
sys.path.insert(0, str(REPO))

from src.evaluation.eval_arc import evaluate_arc

EXPERIMENT_NAME = "exp5_arc_baseline_qwen25_0p5b"

OUTPUT_CSV = RUN_DIR / f"{EXPERIMENT_NAME}_arc.csv"

# Never overwrite a previous evaluation
assert not OUTPUT_CSV.exists(), \
    "Baseline output already exists. Stopping."

assert cfg["model"]["checkpoint_path"] is None, \
    "LoRA checkpoint must be disabled."

print("Running pretrained baseline WITHOUT LoRA.")

evaluate_arc(
    config_path=str(SAFE_CONFIG),
    experiment_name=EXPERIMENT_NAME,
    max_samples=None
)

print("\nBaseline evaluation completed.")
print("Output:", OUTPUT_CSV)


W1116 13:09:01.327000 470 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1116 13:09:01.367000 470 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Running pretrained baseline WITHOUT LoRA.


README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Challenge/train-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B /  190kB            

ARC-Challenge/train-00000-of-00001.parqu(…): downloading bytes:           |  0.00B            

ARC-Challenge/test-00000-of-00001.parque(…): reconstructing file:   0%|          |  0.00B /  204kB            

... (14 outputs left out)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Evaluating ARC [exp5_arc_baseline_qwen25_0p5b]: 100%|██████████| 1172/1172 [02:46<00:00,  7.05it/s]



Baseline evaluation completed.
Output: /content/SC4001_ARC_BASELINE_lp7_9atd/exp5_arc_baseline_qwen25_0p5b_arc.csv


In [11]:

import pandas as pd
import json
import shutil
from pathlib import Path

# Load actual baseline predictions
df = pd.read_csv(OUTPUT_CSV)

assert len(df) == 1172, "Expected 1,172 ARC questions!"
assert "is_correct" in df.columns, "Check CSV columns!"

correct = df["is_correct"]

if correct.dtype == bool:
    correct = correct.astype(int)
else:
    correct = correct.astype(str).str.lower().map({
        "true": 1, "false": 0, "1": 1, "0": 0
    })
    assert correct.notna().all(), "Invalid correctness values"

# Calculate results
summary = {
    "experiment": EXPERIMENT_NAME,
    "dataset": "ARC-Challenge",
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "checkpoint_path": None,
    "n_samples": len(df),
    "correct_answers": int(correct.sum()),
    "accuracy": float(correct.mean())
}

# Generate JSON
json_path = RUN_DIR / f"{EXPERIMENT_NAME}_arc_summary.json"

with json_path.open("x") as f:
    json.dump(summary, f, indent=2)

# Save to Google Drive without overwriting existing files
DEST = Path("/content/drive/MyDrive/SC4001/results")
assert DEST.is_dir(), "Google Drive results folder not found!"

for source in [OUTPUT_CSV, json_path]:
    assert not (DEST / source.name).exists(), \
        f"File already exists: {source.name}"

for source in [OUTPUT_CSV, json_path]:
    with source.open("rb") as src, (DEST / source.name).open("xb") as dst:
        shutil.copyfileobj(src, dst)

print("=== BASELINE RESULTS ===")
print(json.dumps(summary, indent=2))
print("\nBoth files saved to:", DEST)


=== BASELINE RESULTS ===
{
  "experiment": "exp5_arc_baseline_qwen25_0p5b",
  "dataset": "ARC-Challenge",
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "checkpoint_path": null,
  "n_samples": 1172,
  "correct_answers": 401,
  "accuracy": 0.34215017064846415
}

Both files saved to: /content/drive/MyDrive/SC4001/results


In [10]:

import pandas as pd

assert OUTPUT_CSV.is_file(), "Baseline CSV not found."

df = pd.read_csv(OUTPUT_CSV)

print("=== ARC BASELINE RESULTS ===")
print("Total questions:", len(df))
print("Correct answers:", int(df["is_correct"].sum()))
print("Accuracy:", f"{df['is_correct'].mean()*100:.2f}%")

assert len(df) == 1172, \
    "Expected 1,172 ARC-Challenge questions!"

print("\nExisting comparison:")
print("Exp 3 (D1): 33.62%")
print("Exp 4 (D2): 34.39%")


=== ARC BASELINE RESULTS ===
Total questions: 1172
Correct answers: 401
Accuracy: 34.22%

Existing comparison:
Exp 3 (D1): 33.62%
Exp 4 (D2): 34.39%
